In [0]:
import logging
from pyspark.sql.types import StringType
from pyspark.sql import functions as F
from delta.tables import DeltaTable

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("main")

In [0]:
%run ./../../common/utilities

In [0]:
dbutils.widgets.text("catalog", "abcgroup", "Catalog")
dbutils.widgets.text("table", "erp_cust_az12", "Table")

In [0]:
catalog = dbutils.widgets.get("catalog")
table = dbutils.widgets.get("table")

In [0]:
df = (
    spark.table(f"{catalog}.{bronze_schema}.{table}")
)
display(df.limit(5))


In [0]:
df = df.select(
    [
        F.trim(F.col(field.name)).alias(field.name)
        if isinstance(field.dataType, StringType)
        else F.col(field.name)
        for field in df.schema.fields
    ]
)

In [0]:
df = df.withColumn(
    "cid",
    F.when(F.col("cid").startswith("NAS"),
        F.substring(F.col("cid"), 4, F.length(F.col("cid")))
    )
    .otherwise(F.col("cid"))
)

In [0]:
df = (
    df
    .withColumn(
        "bdate",
        F.when(F.col("bdate") > F.current_date(), None)
        .otherwise(F.col("bdate"))
    )
)

In [0]:
df = (
    df
    .withColumn(
        "gen",
        F.when(F.upper(F.col("gen")).isin("F", "FEMALE"), "Female")
        .when(F.upper(F.col("gen")).isin("F", "MALE"), "Male")
        .otherwise("n/a")
    )
 )

In [0]:
COL_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in COL_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
display(df.limit(5))

In [0]:
df_flagged = (
    df.withColumn(
        "is_valid",
        F.when(
            F.col("customer_number").isNotNull() &
            F.col("birth_date").isNotNull() &
            F.col("gender").isNotNull(),
            1
        ).otherwise(0)
    )
)

In [0]:
valid_df = (
    df_flagged
    .filter(F.col("is_valid") == 1)
    .drop("is_valid")
)

invalid_df = (
    df_flagged
    .filter(F.col("is_valid") == 0)
    .drop("is_valid")
)

In [0]:
invalid_df = (
    invalid_df
    .withColumn("error_reason", F.lit("NULL in critical columns"))
    .withColumn("ingestion_ts", F.current_timestamp())
)

In [0]:
invalid_count = invalid_df.count()
if invalid_count > 0:
    logger.warning(f"{invalid_count} invalid records moved to quarantine")

    (
        invalid_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{catalog}.{silver_schema}.{table}_quarantine")
    )

In [0]:
display(valid_df.limit(5))

In [0]:
target_table = f"{catalog}.{silver_schema}.erp_customers"
valid_df.write.mode("overwrite").format("delta").saveAsTable(target_table)

logger.info("Saved table into %s in Delta format.", target_table)

In [0]:
display(valid_df.limit(5))

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {catalog}.{silver_schema}.erp_customers
    LIMIT 5
"""))